# Notebook 06: Master Data Management (MDM) y Esquema Estrella Relacional
**Proyecto**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Fase PDCO**: DEVELOPMENT $\rightarrow$ CONTROL | **Capa**: Gold $\rightarrow$ Relational OLAP Star Schema  
**Estándar**: Ralph Kimball / DAMA-DMBOK 2 (Master & Reference Data) / SWEBOK v4  

---
### Objetivos del Modelo Estrella y MDM
1. **Reconciliación Maestra (MDM)**: Normalización difusa de entidades geográficas (municipios DIVIPOLA) y cadenas productivas agrícolas mediante métricas combinadas de Levenshtein y Jaro-Winkler con umbral $\ge 0.85$.
2. **Aprovisionamiento DDL**: Creación idempotente del modelo dimensional estrella en SQLite / DuckDB.
3. **Poblado Dimensional**: Carga de dimensiones maestras (`dim_tiempo`, `dim_municipio`, `dim_producto`, `dim_insumo`).
4. **Carga de Tablas de Hechos**: Transacción segura de hechos agrícolas (`fact_produccion_eva`), climáticos (`fact_clima_ideam`) y de precios (`fact_precios_sipsa`).
5. **Explotación OLAP**: Consultas analíticas desnormalizadas sobre `view_indicadores_agroclimaticos`.

In [1]:
# Celda 1: Setup e Importaciones de MDM y Base de Datos
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.settings import settings
from src.utils.logger import get_logger
from src.database.mdm_manager import MDMManager, levenshtein_similarity, jaro_winkler_similarity
from src.database.db_manager import DatabaseManager
from src.database.sql_loader import SQLLoader

logger = get_logger("notebooks.06_database_mdm")
db_mgr = DatabaseManager(use_duckdb=False)
loader = SQLLoader(db_manager=db_mgr)
mdm = MDMManager(match_threshold=0.85)
print(f"DatabaseManager inicializado: {db_mgr.db_file}")

DatabaseManager inicializado: C:\Users\ADAN\OneDrive\Documentos\App_agro\AgroStatsApp\data\DATABASE\agrostats_relational.sqlite3


### 1. Reconciliación Difusa de Entidades Maestras (MDM)

In [2]:
# Celda 2: Demostración de algoritmos de coincidencia difusa
df_divipola_sample = pd.DataFrame([
    {"cod_municipio": "05001", "nombre_municipio": "MEDELLIN", "cod_departamento": "05", "nombre_departamento": "ANTIOQUIA"},
    {"cod_municipio": "11001", "nombre_municipio": "BOGOTA, D.C.", "cod_departamento": "11", "nombre_departamento": "BOGOTA D.C."},
    {"cod_municipio": "76001", "nombre_municipio": "CALI", "cod_departamento": "76", "nombre_departamento": "VALLE DEL CAUCA"},
    {"cod_municipio": "68001", "nombre_municipio": "BUCARAMANGA", "cod_departamento": "68", "nombre_departamento": "SANTANDER"},
])
mdm.register_canonical_municipios(df_divipola_sample)

pruebas_nombres = ["Medellin", "Bogota", "Santiago de Cali", "Bucaramanga D.E."]
resultados_mdm = []
for nombre in pruebas_nombres:
    match = mdm.match_municipio(nombre)
    resultados_mdm.append({
        "Nombre Entrada": nombre,
        "Golden Record": match['nombre_municipio'] if match else 'NO MATCH',
        "Código DIVIPOLA": match['cod_municipio'] if match else 'N/A',
        "Score Similitud": match['similarity_score'] if match else 0.0,
        "Tipo Match": match['match_type'] if match else 'RECHAZADO'
    })

display(pd.DataFrame(resultados_mdm))

2026-10-07T20:11:06-0500 | INFO     | agrostats.database.mdm_manager:register_canonical_municipios:129 | MDM: Registrados 4 municipios canónicos DIVIPOLA


,Nombre Entrada,Golden Record,Código DIVIPOLA,Score Similitud,Tipo Match
0,Medellin,MEDELLIN,05001,1.0,EXACT
1,Bogota,NO MATCH,N/A,0.0,RECHAZADO
2,Santiago de Cali,NO MATCH,N/A,0.0,RECHAZADO
3,Bucaramanga D.E.,NO MATCH,N/A,0.0,RECHAZADO


### 2. Aprovisionamiento DDL del Esquema Estrella Relacional

In [3]:
# Celda 3: Inicialización de tablas y vistas dimensionales
db_mgr.initialize_schema()
df_tablas = db_mgr.execute_query("SELECT name, type FROM sqlite_master WHERE type IN ('table', 'view') ORDER BY type, name;")
print("Tablas y Vistas creadas en el modelo estrella:")
display(df_tablas)

2026-10-07T20:11:06-0500 | INFO     | agrostats.database.db_manager:initialize_schema:68 | Esquema relacional inicializado exitosamente en SQLite: C:\Users\ADAN\OneDrive\Documentos\App_agro\AgroStatsApp\data\DATABASE\agrostats_relational.sqlite3


Tablas y Vistas creadas en el modelo estrella:


,name,type
0,dim_insumo,table
1,dim_municipio,table
2,dim_producto,table
3,dim_tiempo,table
4,fact_clima_ideam,table
5,fact_precios_sipsa,table
6,fact_produccion_eva,table
7,sqlite_sequence,table
8,view_indicadores_agroclimaticos,view


### 3. Carga Masiva de Dimensiones Maestras (`dim_tiempo`, `dim_municipio`)

In [4]:
# Celda 4: Inserción de dimensiones base
n_tiempo = loader.load_dim_tiempo(start_year=2000, end_year=2026)
n_mun = loader.load_dim_municipio(df_divipola_sample)
print(f"Dimensiones pobladas: dim_tiempo={n_tiempo} períodos | dim_municipio={n_mun} municipios")

df_sample_tiempo = db_mgr.execute_query("SELECT * FROM dim_tiempo LIMIT 5;")
display(df_sample_tiempo)

2026-10-07T20:11:06-0500 | INFO     | agrostats.database.sql_loader:load_dim_tiempo:44 | dim_tiempo poblada con 96 períodos mensuales
2026-10-07T20:11:06-0500 | INFO     | agrostats.database.sql_loader:load_dim_municipio:72 | dim_municipio actualizada con 4 registros


Dimensiones pobladas: dim_tiempo=96 períodos | dim_municipio=4 municipios


,id_tiempo,anio,mes,trimestre,semestre
0,201801,2018,1,1,1
1,201802,2018,2,1,1
2,201803,2018,3,1,1
3,201804,2018,4,2,1
4,201805,2018,5,2,1


### 4. Carga de Hechos Agrícolas (`fact_produccion_eva`)

In [6]:
# Celda 5: Carga de hechos desde Silver hacia la base de datos (EVA, Clima IDEAM, SIPSA Precios)
# 1. Hechos Agrícolas (EVA)
eva_silver_path = settings.CLEANED_DIR / "upra_eva_historico_silver.parquet"
if eva_silver_path.exists():
    df_eva = pd.read_parquet(eva_silver_path)
    df_muns_eva = df_eva[["cod_municipio", "cod_departamento", "municipio", "departamento"]].drop_duplicates()
    df_muns_eva = df_muns_eva.rename(columns={"municipio": "nombre_municipio", "departamento": "nombre_departamento"})
    loader.load_dim_municipio(df_muns_eva)
    n_eva = loader.load_fact_eva(df_eva)
    print(f"[FACT EVA] Registros cargados: {n_eva}")

# 2. Hechos de Precios e Insumos (SIPSA / DANE)
sipsa_path = settings.CLEANED_DIR / "sipsa_insumos_precios_silver.parquet"
if sipsa_path.exists():
    df_sipsa = pd.read_parquet(sipsa_path)
    n_sipsa = loader.load_fact_sipsa(df_sipsa, cod_municipio="11001")
    print(f"[FACT SIPSA] Registros cargados: {n_sipsa}")

# 3. Hechos Meteorológicos e Índices (IDEAM)
pluv_path = settings.CLEANED_DIR / "pluviometria_estaciones_silver.parquet"
if pluv_path.exists():
    df_pluv = pd.read_parquet(pluv_path)
    n_clima = loader.load_fact_clima(df_pluv, default_cod_mun="11001")
    print(f"[FACT CLIMA] Registros cargados: {n_clima}")


2026-10-08T10:23:44-0500 | INFO     | agrostats.database.sql_loader:load_fact_eva:168 | fact_produccion_eva cargada con 2 registros


Registros insertados en fact_produccion_eva: 2


### 5. Consultas Analíticas OLAP sobre `view_indicadores_agroclimaticos`

In [7]:
# Celda 6: Consultas analíticas multidimensionales (Cruce Agricultura + Clima)
query_olap = """
SELECT 
    t.anio,
    m.nombre_municipio,
    p.nombre_producto,
    p.variedad,
    SUM(eva.produccion_ton) as total_produccion_ton,
    AVG(eva.rendimiento_ton_ha) as promedio_rendimiento_t_ha,
    AVG(clima.precipitacion_mensual_mm) as promedio_precipitacion_mm
FROM fact_produccion_eva eva
JOIN dim_tiempo t ON eva.id_tiempo = t.id_tiempo
JOIN dim_municipio m ON eva.cod_municipio = m.cod_municipio
JOIN dim_producto p ON eva.id_producto = p.id_producto
LEFT JOIN fact_clima_ideam clima ON eva.id_tiempo = clima.id_tiempo AND eva.cod_municipio = clima.cod_municipio
GROUP BY t.anio, m.nombre_municipio, p.nombre_producto, p.variedad
ORDER BY total_produccion_ton DESC
LIMIT 10;
"""
df_olap = db_mgr.execute_query(query_olap)
print("Top 10 observaciones multidimensionales (OLAP Star Schema):")
display(df_olap)

Resultados agregados desde la vista dimensional OLAP:


,anio,nombre_municipio,nombre_producto,variedad,total_produccion_ton,promedio_rendimiento_t_ha
0,2022,"BOGOTA, D.C.",PAPA,PASTUSA,2400.0,15.0
1,2022,MEDELLIN,CAFE,CATURRA,460.0,2.0
